# Redrawing Figs. 2, 3a–c, 4a and SI Fig. S3 for the revision (R1)

This notebook regenerates the figures that the reviewer response requires
(`response/03_implementation_checklist.md`, items **C3**, **C4**, **C5**, **C6**)
**in the formatting of the published figures**, from the flat tables in
`figures/plotdata/`.

The style is taken from the original plotting code, kept in `figures/archive/`:

| Panel | Original style source | What it fixes |
|---|---|---|
| Fig. 2a | `archive/shap_bar_original.ipynb` + `shap.summary_plot(plot_type='bar')` | horizontal bars, SHAP blue `#008bfb`, no *y* labels (shared with **b**) |
| Fig. 2b | `archive/KPS_DFT_free_BO_SGK_original.ipynb`, cell 9 — `shap.summary_plot(..., cmap='jet', plot_type='dot')` | beeswarm, `jet` colour map, `Feature value` Low→High colour bar |
| Fig. 3a–c | `archive/KPS_DFT_free_BO_SGK_original.ipynb`, cell 24 | scatter coloured by SHAP, `bwr`, symmetric limits, log *x* |
| Fig. 4a | same notebook, cell 28 family | scatter coloured by MPF, `jet` |
| SI Fig. S3 | `groupkfold_full_oof_shap_hp/plotresuult.ipynb` | black points, red dashed 1:1 line, rounded metric box |

Common to all of them: `font.family = 'Times New Roman'`, `mathtext.fontset = 'cm'`,
bold `(a)`, `(b)`, … panel letters set outside the axes.

**What changed in the numbers** (not in the style): SHAP is now reported in
$\log(1+x)$ space and computed on held-out folds only, averaged over 20 seeds.
See `figures/README.md` and `figures/plotdata/README.md`.

The beeswarm is drawn with SHAP's own point-stacking algorithm reimplemented below, so the
`shap` package is **not** required — only `numpy`, `pandas` and `matplotlib`.


In [ ]:
# ============================================================
# 0) Setup — paths, fonts, shared style
# ============================================================
import os, sys, gc, glob
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.ticker import LogLocator, FixedLocator, FixedFormatter, NullFormatter
from matplotlib import font_manager

# --- locate plotdata/ whether this notebook sits in figures/ or in figures/plotdata/ ---
HERE = os.path.abspath(os.getcwd())
for cand in (os.path.join(HERE, 'plotdata'), HERE,
             os.path.join(HERE, '..', 'plotdata'),
             os.path.join(HERE, 'figures', 'plotdata')):
    if os.path.exists(os.path.join(cand, 'fig2a_shap_ranking.csv')):
        PLOTDATA = os.path.abspath(cand)
        break
else:
    raise FileNotFoundError('plotdata/fig2a_shap_ranking.csv not found — run this notebook '
                            'from deposit/figures/ or deposit/figures/plotdata/')

OUTDIR = os.path.abspath(os.path.join(PLOTDATA, '..', 'redrawn'))
os.makedirs(OUTDIR, exist_ok=True)
print('plotdata :', PLOTDATA)
print('output   :', OUTDIR)

# --- fonts: the published figures use Times New Roman + Computer Modern mathtext ---
# Times New Roman is not redistributable, so it is frequently missing. Instead of
# requiring a system-wide install, register any times*.ttf that is reachable from this
# machine straight into matplotlib's font manager - no sudo, no ~/.cache/matplotlib
# clearing, and it takes effect the moment this cell is re-run.
FONT_DIRS = [
    os.environ.get('TIMES_FONT_DIR', ''),                  # explicit override
    os.path.join(HERE, 'fonts'),                           # fonts/ next to this notebook
    os.path.join(PLOTDATA, '..', 'fonts'),                 # figures/fonts/
    os.path.expanduser('~/.fonts'),
    os.path.expanduser('~/.local/share/fonts'),
    '/usr/share/fonts/truetype/msttcorefonts',
    '/Library/Fonts', '/System/Library/Fonts/Supplemental',  # macOS
    '/mnt/c/Windows/Fonts',                                  # WSL
    '/media/sf_Windows/Fonts',                               # VirtualBox shared folder
    'C:\\Windows\\Fonts',                                     # native Windows
]
for _d in FONT_DIRS:
    if not _d or not os.path.isdir(_d):
        continue
    for _p in sorted(glob.glob(os.path.join(_d, '[Tt]imes*.tt[fc]'))):
        try:
            font_manager.fontManager.addfont(_p)
        except Exception:
            pass        # .ttc collections are not readable by every matplotlib build

# Liberation Serif and Nimbus Roman are both *metrically* compatible with Times New
# Roman, so line breaks and figure sizes are unchanged either way; Liberation Serif is
# the closer match in glyph shape, so it is preferred. Swap the two names below if you
# would rather keep the URW/Adobe-Times look.
_avail = {f.name for f in font_manager.fontManager.ttflist}
for SERIF in ('Times New Roman', 'Liberation Serif', 'Nimbus Roman',
              'DejaVu Serif', 'serif'):
    if SERIF == 'serif' or SERIF in _avail:
        break
if SERIF != 'Times New Roman':
    print(f'[note] "Times New Roman" not found; using the metric-compatible "{SERIF}". '
          'To reproduce the published figures glyph-for-glyph, copy times.ttf, '
          'timesbd.ttf, timesi.ttf and timesbi.ttf into a "fonts/" folder next to this '
          'notebook (or point TIMES_FONT_DIR at them) and re-run this cell.')

plt.rcParams['font.family']        = SERIF
plt.rcParams['mathtext.fontset']   = 'cm'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype']       = 42     # editable text in the PDF
plt.rcParams['ps.fonttype']        = 42
plt.rcParams['savefig.bbox']       = 'tight'

# --- sizes used throughout (the published figures are ~13-15 pt at final size) ---
FS_TICK, FS_LABEL, FS_PANEL, FS_CBAR = 13, 15, 17, 13

def panel_letter(ax, s, x=-0.14, y=1.05, fontsize=FS_PANEL):
    """Bold '(a)' outside the top-left corner, as in the published figures."""
    ax.text(x, y, s, transform=ax.transAxes, ha='left', va='bottom',
            fontsize=fontsize, fontweight='bold')

# ---- output resolution -------------------------------------------------------
# Fig. 2 is raised furthest so the individual beeswarm points stay resolved.
# The scatter layers are rasterized inside the PDF, so the dpi is passed to the PDF
# save as well - otherwise the vector file would embed low-resolution bitmaps.
DPI = {'figure2': 1000, 'figure3abc': 800, 'figure4a': 800, 'SI_3': 800}
DPI_DEFAULT = 800

# Memory. Everything below exists because these figures are large and this pipeline is
# expected to run on a modest machine. Measured peaks for Fig. 3a-c (5.4 x 11.4 in):
#
#     building the figure                    0.10 GB
#     writing the png at 800 dpi             0.45 GB
#     writing the pdf at 800 dpi             1.18 GB   <- the expensive one
#     writing the pdf at 400 dpi             0.39 GB
#
# The pdf dominates because the scatter layers are `rasterized=True`: matplotlib renders
# each of them into a full-canvas buffer at the save dpi before embedding it. Text, axes
# and lines stay vector whatever this is set to, so the only thing PDF_RASTER_DPI governs
# is the resolution of the embedded point clouds. 400 dpi is above the 300 dpi that
# journals ask for raster content and cuts peak memory by a factor of three.
PDF_RASTER_DPI = 400        # dpi of the rasterized layers inside the pdf

# The png is a single RGBA canvas of w*h*dpi^2*4 bytes. Fig. 2 at 1000 dpi is
# 13000 x 7200 px = 94 Mpx, about 0.4 GB per buffer. Past this cap the png dpi is
# reduced rather than risking the OS killing the process, which it does before Python
# can catch anything. The pdf is unaffected. None disables the cap.
PNG_MAX_MPIX = 200          # megapixels; None disables the cap

LAYOUT_DPI = 100      # dpi used only to measure the crop box
PAD_INCHES = 0.1      # matplotlib's own default padding for bbox_inches='tight'

def tight_bbox(fig):
    """Measure the tight crop box once, cheaply.

    `bbox_inches='tight'` makes matplotlib render the whole canvas an EXTRA time at the
    save dpi purely to find where the ink ends, and it does that once per output file.
    At 1000 dpi those extra passes are most of the memory cost. The crop box is a
    geometric quantity in inches, so measuring it at 100 dpi gives the same answer for
    1/100 of the pixels; the resulting Bbox is then handed to savefig, which skips its
    own tight pass. Peak memory drops by roughly half and nothing about the output
    changes.
    """
    dpi0 = fig.dpi
    try:
        fig.set_dpi(LAYOUT_DPI)
        bb = fig.get_tightbbox(fig.canvas.get_renderer())
    finally:
        fig.set_dpi(dpi0)
    return bb.padded(PAD_INCHES)

def save(fig, stem, dpi=None):
    dpi = dpi if dpi is not None else DPI.get(stem, DPI_DEFAULT)
    w, h = fig.get_size_inches()
    print(f'{stem}: {w:.1f} x {h:.1f} in')
    bbox = tight_bbox(fig)
    for ext in ('pdf', 'png'):
        d = PDF_RASTER_DPI if ext == 'pdf' else dpi
        if ext == 'png' and PNG_MAX_MPIX:
            mpix = w * h * d * d / 1e6
            if mpix > PNG_MAX_MPIX:
                d = int((PNG_MAX_MPIX * 1e6 / (w * h)) ** 0.5)
                print(f'  [note] png at {dpi} dpi would be {mpix:.0f} Mpx '
                      f'(~{mpix*4/1000:.1f} GB to render); capped to {d} dpi by '
                      f'PNG_MAX_MPIX.')
        p = os.path.join(OUTDIR, f'{stem}.{ext}')
        fig.savefig(p, dpi=d, bbox_inches=bbox)
        gc.collect()
        print(f'  written: {p}   ({d} dpi, {int(w*d):,} x {int(h*d):,} px, '
              f'{os.path.getsize(p)/1e6:.1f} MB)')
    plt.close(fig)
    gc.collect()

# --- manuscript symbols for the descriptor names (matches Table 1 / Fig. 2) ---
SYMBOL = {
    'p_metric'        : 'MSBI',
    'packing_fraction': 'MPF',
    'pd_ratio'        : r'$p/d$ electron ratio',
    'center_std_angle': r'$XMX_{\mathrm{std}}$',
    'p_metric_std'    : r'$\sigma_{\mathrm{inhom}}$',
    'labelled_1st'    : r'$d_{CC}^{(1)}$',
    'p_orb_e_non'     : r'$n_{p,X}$',
    'd_lone_pair'     : r'$n_{\mathrm{unpaired}}$',
    'd_orb_e'         : r'$n_{d,M}$',
    'center_avg_angle': r'$XMX_{\mathrm{avg}}$',
    'global_1st'      : r'$d_{MM}^{(1)}$',
    'center_max_angle': r'$XMX_{\mathrm{max}}$',
    'proxy_M_magnet'  : r'$\mu_{\mathrm{spin}}$',
    'avg_delta'       : r'$\langle\delta\rangle$',
}
def sym(feature, fallback=None):
    return SYMBOL.get(feature, fallback if fallback is not None else feature.replace('_', ' '))

SHAP_BLUE = '#008bfb'      # the blue of shap.summary_plot(plot_type='bar')
print('matplotlib', mpl.__version__, '| font:', SERIF)

---
## Fig. 2 — SHAP bars + beeswarm  (checklist **C3**)

`fig2a_shap_ranking.csv` → panel **a**, `fig2b_beeswarm_long.csv` → panel **b**.

Both panels are in $\log(1+x)$ units; the eV conversion by local linearization has been
withdrawn. The feature labels sit between the two panels, exactly as in the published
figure — panel **a** carries no *y* tick labels.


In [ ]:
# ============================================================
# 1) Fig. 2 — (a) mean|SHAP| bars, (b) beeswarm
# ============================================================
NTOP = 12

rank = pd.read_csv(os.path.join(PLOTDATA, 'fig2a_shap_ranking.csv'))
bees = pd.read_csv(os.path.join(PLOTDATA, 'fig2b_beeswarm_long.csv'))
top  = rank.head(NTOP).reset_index(drop=True)
labels = [sym(f) for f in top.feature]

print(f'Sigma mean|SHAP| (all {len(rank)} descriptors) = {rank.mean_abs_shap.sum():.4f}  [log(1+x)]')
print(f'top three  = {top.share_pct.head(3).sum():.2f} %')
print(f'top twelve = {top.cumulative_pct.iloc[-1]:.2f} % (cumulative)')


def shap_beeswarm_offsets(values, row_height=0.4, nbins=100, seed=0):
    """SHAP's point-stacking rule (shap/plots/_beeswarm.py), reimplemented.

    Points are binned by their x position and then stacked outward from the row
    centre, which is what produces the characteristic violin-like silhouette.
    """
    values = np.asarray(values, float)
    rng = np.random.default_rng(seed)
    span = values.max() - values.min()
    quant = np.round(nbins * (values - values.min()) / (span + 1e-12))
    order = np.argsort(quant + rng.normal(0, 1e-6, values.size))
    ys, layer, last = np.zeros(values.size), 0, -1
    for i in order:
        if quant[i] != last:
            layer = 0
        ys[i] = np.ceil(layer / 2) * ((layer % 2) * 2 - 1)
        layer += 1
        last = quant[i]
    return ys * (0.9 * (row_height / (np.max(np.abs(ys)) + 1)))


# 13.0 x 7.55 in: the cropped output then carries the aspect of the published Fig. 2
# (1.555 wide-to-tall). Its text is already the published size relative to the figure.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.0, 7.55),
                               gridspec_kw=dict(width_ratios=[1.0, 1.35], wspace=0.42))

# ---------------- (a) bars ----------------
ypos = np.arange(NTOP)[::-1]                       # rank 1 at the top
# white bars with a black outline (the SHAP_BLUE fill of the published panel is
# kept in the setup cell should it be wanted back: color=SHAP_BLUE, linewidth=0)
ax1.barh(ypos, top.mean_abs_shap, height=0.70,
         facecolor='white', edgecolor='black', linewidth=1.2)
ax1.axvline(0, color='0.6', lw=0.9, zorder=0)
ax1.set_yticks(ypos); ax1.set_yticklabels([])
ax1.set_ylim(-0.7, NTOP - 0.3)
ax1.set_xlim(0, top.mean_abs_shap.max() * 1.06)
ax1.set_xlabel(r'Mean absolute SHAP value  [$\log(1+x)$]', fontsize=FS_LABEL)
ax1.tick_params(axis='x', labelsize=FS_TICK)
ax1.tick_params(axis='y', length=0)
for sp in ('top', 'right', 'left'):
    ax1.spines[sp].set_visible(False)
panel_letter(ax1, '(a)', x=-0.06)

# --- mean|SHAP| written on each bar, right-aligned inside it -----------------------
# Same numbers, same format as the Table 1 rows regenerated in the last cell.
# The bottom bars are only ~7 % as long as the top one and cannot hold the number,
# so any label that would spill past the bar's left edge is moved just outside the
# bar end instead. All of them are black either way.
# Set BAR_VALUES = False to drop the annotation entirely.
BAR_VALUES, BAR_VALUE_FMT, BAR_VALUE_FS = True, '{:.4f}', FS_TICK
if BAR_VALUES:
    _pad = 0.012 * ax1.get_xlim()[1]
    _tx = [ax1.text(v - _pad, yp_, BAR_VALUE_FMT.format(v), ha='right', va='center',
                    fontsize=BAR_VALUE_FS, color='black', zorder=3)
           for yp_, v in zip(ypos, top.mean_abs_shap)]
    fig.canvas.draw()                      # text extents are only known once drawn
    _r = fig.canvas.get_renderer()
    _bar_x0 = ax1.transData.transform((0, 0))[0]
    for _t, _v in zip(_tx, top.mean_abs_shap):
        if _t.get_window_extent(renderer=_r).x0 < _bar_x0 + 2:
            _t.set_position((_v + _pad, _t.get_position()[1]))
            _t.set_ha('left')

# ---------------- (b) beeswarm ----------------
CMAP_BEE = 'jet'                                   # cmap='jet' in the original cell 9
for k, feat in enumerate(top.feature):
    g = bees[bees.feature == feat]
    v = g.shap.to_numpy(float)
    f = g.feature_value.to_numpy(float)
    lo, hi = np.nanpercentile(f, [5, 95])           # shap's default colour clipping
    c = np.clip((f - lo) / (hi - lo + 1e-12), 0, 1)
    y = ypos[k] + shap_beeswarm_offsets(v, seed=k)
    ax2.axhline(ypos[k], color='0.85', lw=0.6, ls=(0, (1, 3)), zorder=0)
    ax2.scatter(v, y, c=c, cmap=CMAP_BEE, norm=Normalize(0, 1),
                s=6, linewidths=0, alpha=1.0, rasterized=True, zorder=2)

ax2.axvline(0, color='0.6', lw=0.9, zorder=1)
ax2.set_yticks(ypos)
ax2.set_yticklabels(labels, fontsize=FS_LABEL)
ax2.set_ylim(-0.7, NTOP - 0.3)
ax2.set_xlabel(r'Mean SHAP value  [$\log(1+x)$]', fontsize=FS_LABEL)
ax2.tick_params(axis='x', labelsize=FS_TICK)
ax2.tick_params(axis='y', length=0)
for sp in ('top', 'right', 'left'):
    ax2.spines[sp].set_visible(False)
panel_letter(ax2, '(b)', x=-0.0)

cb = fig.colorbar(plt.cm.ScalarMappable(norm=Normalize(0, 1), cmap=CMAP_BEE),
                  ax=ax2, pad=0.02, fraction=0.045, aspect=32)
cb.set_ticks([0, 1]); cb.set_ticklabels(['Low', 'High'])
cb.ax.tick_params(labelsize=FS_CBAR, length=0)
cb.set_label('Feature value', fontsize=FS_CBAR)
cb.outline.set_visible(False)

# --- place the feature names in the gutter between (a) and (b) -------------------
# The names stay ax2's y-tick labels, so they track the axes; ha='center' plus a pad
# measured off the real gutter width puts each name on the same vertical line.
# NAME_X is where that line sits across the gutter:
#     0.0 = flush against panel (a)   0.5 = midway   1.0 = flush against panel (b)
# 1.0 reproduces matplotlib's default right-aligned tick labels.
# Run after the colorbar, which takes its width out of ax2's right side.
NAME_X = 0.5
_p1, _p2 = ax1.get_position(), ax2.get_position()
_gutter_pt = (_p2.x0 - _p1.x1) * fig.get_size_inches()[0] * 72.0
ax2.tick_params(axis='y', pad=_gutter_pt * (1.0 - NAME_X))
for _lab in ax2.get_yticklabels():
    _lab.set_horizontalalignment('center')

save(fig, 'figure2')
plt.show()

---
## Fig. 3 a–c — SSE vs MSBI / MPF / $p/d$, coloured by SHAP  (checklist **C4**)

Panels **d–e** (CuO and FeSi structures and bands) do not involve SHAP and are unchanged —
they come from `archive/plotband_CuO_original.ipynb` and `archive/plotband_FeSi_original.ipynb`.
This cell therefore writes the **a–c column only**, in the stacked layout of the published
figure, ready to be composed with **d–e**.

Colour limits are symmetric about 0 and set per panel, as in the published figure
($\pm 0.3$ / $\pm 0.1$ / $\pm 0.1$ in the old eV scale). The defaults below are the
corresponding round numbers in $\log(1+x)$ units; edit `CLIM` to taste.


In [ ]:
# ============================================================
# 2) Fig. 3 a-c — SSE vs descriptor, coloured by that descriptor's SHAP
# ============================================================
sc3 = pd.read_csv(os.path.join(PLOTDATA, 'fig3abc_scatter.csv'))

PANELS = [
    # (x column, shap column, x label, panel letter, colour limit, x ticks, x limits)
    ('msbi',     'msbi_shap',     'MSBI',                  '(a)', 0.10, None,         None),
    ('mpf',      'mpf_shap',      'MPF',                   '(b)', 0.10, None,         None),
    ('pd_ratio', 'pd_ratio_shap', r'$p/d$ electron ratio', '(c)', 0.10, [0.1, 0.5, 1, 2], (0.1, 2.8)),
]
CMAP_SHAP = 'bwr'          # the original used plt.get_cmap('bwr')

# Text size for this panel stack, relative to the shared sizes the other figures use.
# 1.0 - the default - is the size the published panels carry. The layout adapts: past
# BIG3 the colorbar label no longer fits along the panel height and is broken over two
# lines, matplotlib thins the y ticks so they are pinned, and the gutters grow.
# The canvas below puts each panel at the published size, and at that size the shared
# FS_* values already give the published text - 16.6 pt over a 2.77 in panel.
FS3_SCALE = 1.2
BIG3 = FS3_SCALE > 1.45
F_TICK, F_LABEL, F_PANEL, F_CBAR = (s * FS3_SCALE
                                    for s in (FS_TICK, FS_LABEL, FS_PANEL, FS_CBAR))
# Panel letters are set in absolute points, not scaled with FS*_SCALE, so that every
# figure carries them at the same size once assembled. The composite places this
# panel at scale 1.05, so 19.05 pt here renders as the 20 pt the published figures use.
PANEL_PT = 19.05
# Extreme-MSBI case studies annotated on panel (a). The row is picked as the closest
# match to the (MSBI, SSE) pair quoted in the caption, and the filename chosen is printed
# below so it can be checked; edit the targets or the label offsets as needed.
# Offsets are in points from the marked point. FeSi sits near the left of the log
# abscissa, so a mostly-horizontal offset runs its label through the left spine and into
# the y tick labels; the published panel puts it above the point instead.
ANNOTATE  = {'CuO' : dict(target=(1.018, 1.634),   offset=(-62, -20)),
             'FeSi': dict(target=(0.0003, 0.003),  offset=(-30,  78))}

# Each published a-c panel measures 2.765 x 2.16 in (1.28 wide-to-tall), measured off
# the axes rectangle in a 200 dpi render of the published PDF. This canvas reproduces
# it - the margins around the axes are set by the text and are about 1.2 x 3.6 in.
fig, axes = plt.subplots(3, 1, figsize=(3.85, 10.69))

for ax, (xcol, ccol, xlab, tag, clim, xticks, xlim) in zip(axes, PANELS):
    x = sc3[xcol].to_numpy(float)
    y = sc3['sse'].to_numpy(float)
    c = sc3[ccol].to_numpy(float)
    m = x > 0                                   # log axis
    if (~m).sum():
        print(f'  {xcol}: {(~m).sum()} rows with x = 0 excluded from the log axis')

    sc = ax.scatter(x[m], y[m], c=c[m], cmap=CMAP_SHAP,
                    norm=TwoSlopeNorm(vcenter=0.0, vmin=-clim, vmax=clim),
                    s=12, linewidths=0, alpha=1.0, rasterized=True)
    ax.set_xscale('log')
    ax.set_xlabel(xlab, fontsize=F_LABEL)
    ax.set_ylabel('SSE (eV)', fontsize=F_LABEL)
    ax.tick_params(labelsize=F_TICK)
    # the published panels step the ordinate every 0.2; left to itself matplotlib
    # thins it to 0.5, and further to 0 / 1 once the tick font grows
    _yt = np.arange(0.0, 1.61, 0.2)
    ax.yaxis.set_major_locator(FixedLocator(_yt))
    ax.yaxis.set_major_formatter(FixedFormatter([f'{t:.1f}' for t in _yt]))
    if xticks is not None:
        ax.xaxis.set_major_locator(FixedLocator(xticks))
        ax.xaxis.set_major_formatter(FixedFormatter([f'{t:g}' for t in xticks]))
        ax.xaxis.set_minor_formatter(NullFormatter())
    else:
        # numticks high enough that matplotlib labels every decade, as the published
        # panels do, instead of thinning to every second one
        ax.xaxis.set_major_locator(LogLocator(base=10, numticks=20))
        ax.xaxis.set_minor_formatter(NullFormatter())
    if xlim is not None:
        ax.set_xlim(*xlim)
    # Decade sub-ticks (2...9) so the reader can see the abscissa is logarithmic.
    # Panels (a) and (b) span six and three decades, and over that range matplotlib's
    # automatic minor locator returns nothing, which is why (a) had no sub-ticks at all.
    ax.xaxis.set_minor_locator(LogLocator(base=10, subs=tuple(range(2, 10)),
                                          numticks=100))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.tick_params(axis='both', which='major', length=5.5 * FS3_SCALE, width=1.0)
    ax.tick_params(axis='x',    which='minor', length=3.0 * FS3_SCALE, width=0.8)
    panel_letter(ax, tag, x=-0.26 if BIG3 else -0.20, fontsize=PANEL_PT)

    cb = fig.colorbar(sc, ax=ax, pad=0.02, fraction=0.045, aspect=18)
    # the published colorbars are labelled; the revision reports mean SHAP in log(1+x)
    # space rather than the eV of the published panel, so the unit differs
    cb.set_label(r'Mean SHAP Value  [$\log(1+x)$]', labelpad=4,
                 fontsize=(FS_CBAR - 1) * FS3_SCALE)
    cb.ax.tick_params(labelsize=(FS_CBAR - 2) * FS3_SCALE)

    # --- CuO / FeSi call-outs on panel (a), as published ---
    if tag == '(a)':
        for name, spec in ANNOTATE.items():
            tx, ty = spec['target']; off = spec['offset']
            # nearest row in (log10 MSBI, SSE), so the match is scale-free in x
            d = (np.log10(sc3.msbi.to_numpy(float)) - np.log10(tx)) ** 2 + \
                ((sc3.sse.to_numpy(float) - ty) / max(ty, 1e-3)) ** 2
            row = sc3.iloc[[int(np.argmin(d))]]
            xa, ya = float(row[xcol].iloc[0]), float(row['sse'].iloc[0])
            print(f'  {name}: {row.filename.iloc[0]}  (MSBI = {xa:.4g}, SSE = {ya:.4g} eV)')
            ax.scatter([xa], [ya], s=170, facecolors='none', edgecolors='k',
                       linewidths=1.6, zorder=5)
            # the ring is ~0.2 in across, which at the automatic limits overruns the
            # top and right spines for the CuO point; give it room on both
            ax.set_ylim(top=max(ax.get_ylim()[1], 1.80))
            ax.set_xlim(right=max(ax.get_xlim()[1], xa * 1.9))
            ax.annotate(name, xy=(xa, ya), xytext=off, textcoords='offset points',
                        fontsize=F_LABEL, ha='center', va='center',
                        arrowprops=dict(arrowstyle='->', lw=1.4, color='k'), zorder=6)

fig.subplots_adjust(hspace=0.55 if BIG3 else 0.38)
save(fig, 'figure3abc')
plt.show()

print('\npanels d-e (CuO, FeSi structures and bands) are unaffected and are kept as published:')
print('  archive/plotband_CuO_original.ipynb, archive/plotband_FeSi_original.ipynb')

---
## Fig. 4a — packing scatter  (checklist **C5**)

The checklist asks only that the abscissa be **confirmed** to be `labelled_1st` = $d_{CC}^{(1)}$.
The cell redraws the panel in the published style and prints the window statistics on
**both** candidate abscissae so the caption number can be checked against the figure.


In [ ]:
# ============================================================
# 3) Fig. 4a — SSE vs d_CC^(1), coloured by MPF   (+ C5 abscissa check)
# ============================================================
sc4 = pd.read_csv(os.path.join(PLOTDATA, 'fig4a_packing_scatter.csv'))

XCOL      = 'd_CC_1'      # 'labelled_1st' — distance between the two magnetic sites
MPF_VMAX  = 0.30          # colour-bar top, as published
HI_SSE    = 0.8           # "high SSE" in the caption
MPF_CUT, D_CUT = 0.20, 3.5

# ---- C5: which column reproduces the caption numbers? ----
print('Window:  MPF > %.2f  and  x < %.1f A      (high SSE = SSE > %.1f eV)\n'
      % (MPF_CUT, D_CUT, HI_SSE))
hi = sc4.sse.to_numpy(float) > HI_SSE
for col, name in (('d_CC_1', 'labelled_1st = d_CC^(1)'), ('d_MM_1', 'global_1st = d_MM^(1)')):
    w = (sc4.mpf.to_numpy(float) > MPF_CUT) & (sc4[col].to_numpy(float) < D_CUT)
    print(f'  {name:26s} range {sc4[col].min():.2f}-{sc4[col].max():.2f} A | '
          f'window holds {100*w.mean():5.1f} % of entries, '
          f'{100*w[hi].mean():5.1f} % of high-SSE  ->  enrichment {w[hi].mean()/w.mean():.2f}x')
print('\n  labelled_1st >= global_1st for all rows:',
      bool((sc4.d_CC_1.to_numpy(float) >= sc4.d_MM_1.to_numpy(float) - 1e-9).all()))
print('  -> the abscissa of the published Fig. 4a is labelled d_CC^(1); the caption numbers\n'
      '     38.5 % / 76.2 % / 1.98x belong to d_MM^(1), while d_CC^(1) gives 33.1 % / 68.3 %\n'
      '     / 2.07x, which are the values checklist item A6 asks for. Keep the two consistent.')

# ---- the panel ----
# Text size for this panel, relative to the shared sizes the other figures use.
# 1.0 - the default - is the size the published panel carries. Past BIG4 the 0.5 A
# tick labels collide, so only whole angstroms are labelled and the 0.5 positions
# become unlabelled minor ticks.
# 1.54: the published panel carries 40.1 pt text over a 3.90 in panel, 10.3 pt per inch
FS4_SCALE = 1.54
BIG4 = FS4_SCALE > 1.8
F_TICK, F_LABEL, F_PANEL, F_CBAR = (s * FS4_SCALE
                                    for s in (FS_TICK, FS_LABEL, FS_PANEL, FS_CBAR))
# Panel letters are set in absolute points, not scaled with FS*_SCALE, so that every
# figure carries them at the same size once assembled. The composite places this
# panel at scale 1.0, so 20.00 pt here renders as the 20 pt the published figures use.
PANEL_PT = 20.0
# The published Fig. 4a panel measures 3.900 x 3.070 in (1.270 wide-to-tall). An
# earlier pass matched 0.98 by mistake - that is the neighbouring Fig. 4b panel, which
# is square. This canvas reproduces 4a.
fig, ax = plt.subplots(figsize=(5.40, 4.02))
# Marker *diameter* scale. matplotlib's `s` is an area, so doubling the diameter
# means multiplying s by PT_SCALE**2. 1.0 is the published size.
PT_SCALE = 1.0
# 'rainbow', not 'jet': sampling the published colorbar gives (91, 12, 198) at its
# foot, the violet that rainbow starts on; jet starts at navy (0, 0, 127).
sc = ax.scatter(sc4[XCOL], sc4['sse'], c=sc4['mpf'], cmap='rainbow',
                vmin=0.0, vmax=MPF_VMAX, s=12 * PT_SCALE ** 2,
                linewidths=0, alpha=1.0, rasterized=True)
ax.set_xlabel(r'$d_{CC}^{(1)}$  ($\mathrm{\AA}$)' if XCOL == 'd_CC_1'
              else r'$d_{MM}^{(1)}$  ($\mathrm{\AA}$)', fontsize=F_LABEL)
ax.set_ylabel('SSE (eV)', fontsize=F_LABEL)
ax.tick_params(labelsize=F_TICK)
# the published panel steps the ordinate every 0.2, as Fig. 3 does
_yt = np.arange(0.0, 1.61, 0.2)
ax.set_yticks(_yt)
ax.set_yticklabels([f'{t:.1f}' for t in _yt])
ax.set_xlim(2.2, 6.5)
ax.set_xticks(np.arange(2.5, 6.51, 1.0 if BIG4 else 0.5))
if BIG4:
    ax.set_xticks(np.arange(2.5, 6.51, 0.5), minor=True)
    ax.tick_params(axis='x', which='minor', length=4.5, width=1.0)
ax.tick_params(axis='both', which='major', length=5.5 * FS4_SCALE, width=1.0)
# -0.24 puts the letter at the very left of the page, where the composite can put
# panel b's letter too - b's y tick labels leave no room further right
panel_letter(ax, '(a)', x=-0.22 if BIG4 else -0.24, fontsize=PANEL_PT)

# points above MPF_VMAX saturate at the top colour, as in the published panel
cb = fig.colorbar(sc, ax=ax, pad=0.02, fraction=0.046, aspect=20)
cb.set_label('MPF', fontsize=F_CBAR)
cb.set_ticks(np.arange(0.0, MPF_VMAX + 1e-9, 0.05))   # as published, not thinned by the
cb.ax.set_yticklabels([f'{t:.2f}' for t in np.arange(0.0, MPF_VMAX + 1e-9, 0.05)])
cb.ax.tick_params(labelsize=(FS_CBAR - 1) * FS4_SCALE)   # larger font

save(fig, 'figure4a')
plt.show()

---
## SI Fig. S3 — parity plot + decile bias  (checklist **C6**)

Panel **a** keeps the published parity style (black points, red dashed 1:1 line, rounded
metric box). Panel **b** is the new decile-resolved bias the response promises.

The points are the **seed-averaged** out-of-fold predictions, so the $R^2$ printed here
($0.7168$) is not the headline $0.6933 \pm 0.0171$, which pools each seed separately —
the Supplementary Information states both.


In [ ]:
# ============================================================
# 4) SI Fig. S3 — (a) parity plot, (b) decile-resolved prediction bias
# ============================================================
oof = pd.read_csv(os.path.join(PLOTDATA, 'si3_oof_predictions.csv'))
dec = pd.read_csv(os.path.join(PLOTDATA, 'si3_decile.csv'))

y  = oof.sse_dft.to_numpy(float)
yp = oof.sse_pred.to_numpy(float)
r2  = 1 - ((y - yp) ** 2).sum() / ((y - y.mean()) ** 2).sum()
mae = np.abs(y - yp).mean() * 1000
print(f'n = {len(y)}   R2(eV) = {r2:.4f}   MAE = {mae:.1f} meV   (seed-averaged predictions)')
print(f'decile bias: 1st {dec.bias_pct.iloc[0]:+.1f} %  ->  10th {dec.bias_pct.iloc[-1]:+.1f} %')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.4, 5.0),
                               gridspec_kw=dict(width_ratios=[1.0, 1.05], wspace=0.30))

# ---------------- (a) parity, in the published style ----------------
LIM   = 1.8
TICKS = np.arange(0.0, LIM + 1e-9, 0.3)
ax1.plot([0, LIM], [0, LIM], 'r--', lw=1.4, alpha=0.85, zorder=0)
# NOT rasterized: at 800 dpi the rasterized layer of this panel is written into the
# PDF as a strip-split image and poppler renders visible horizontal seams through
# the point cloud. 3,845 points draw fine as vector, and the PDF stays small.
ax1.scatter(y, yp, s=22, color='k', alpha=1.0, linewidths=0, rasterized=False)
ax1.set_xlim(0, LIM); ax1.set_ylim(0, LIM)
ax1.set_aspect('equal', 'box')
ax1.set_xticks(TICKS); ax1.set_yticks(TICKS)
ax1.set_xlabel('Actual SSE (eV)', fontsize=FS_LABEL)
ax1.set_ylabel('Predicted SSE (eV)', fontsize=FS_LABEL)
ax1.tick_params(labelsize=FS_TICK)
ax1.text(0.04, 0.96, '\n'.join((rf'$R^2 = {r2:.2f}$', rf'MAE = {mae:.1f} meV')),
         transform=ax1.transAxes, va='top', ha='left', fontsize=FS_TICK,
         bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', alpha=0.9))
panel_letter(ax1, '(a)', x=-0.18)

# ---------------- (b) decile bias ----------------
bars = ax2.bar(dec.decile, dec.bias_pct, width=0.72,
               color=['#c1272d' if b < 0 else '#1f5fa8' for b in dec.bias_pct],
               edgecolor='black', linewidth=0.7)
ax2.axhline(0, color='k', lw=0.9)
pad = 0.045 * (dec.bias_pct.max() - dec.bias_pct.min())
for d, b in zip(dec.decile, dec.bias_pct):
    ax2.text(d, b + (pad if b >= 0 else -pad), f'{b:+.0f}', ha='center',
             va='bottom' if b >= 0 else 'top', fontsize=FS_TICK - 3)
ax2.set_xticks(dec.decile)
ax2.set_xlabel('SSE decile', fontsize=FS_LABEL)
ax2.set_ylabel('Mean prediction bias (%)', fontsize=FS_LABEL)
ax2.tick_params(labelsize=FS_TICK)
ax2.margins(y=0.14)
for sp in ('top', 'right'):
    ax2.spines[sp].set_visible(False)
panel_letter(ax2, '(b)', x=-0.16)

save(fig, 'SI_3')
plt.show()

---
## Table 1 — the LaTeX rows that go with Fig. 2a  (checklist **A9**)

Already applied in `Draft_npj_R1.tex`; this cell regenerates the rows from the same CSV so
the table and the figure can be checked against one source.


In [ ]:
# ============================================================
# 5) Table 1 rows (top 12), regenerated from fig2a_shap_ranking.csv
# ============================================================
tot = rank.mean_abs_shap.sum()
print(f'% Sigma_k mean|SHAP|_k = {tot:.4f}  [log(1+x)]')
for _, r in rank.head(NTOP).iterrows():
    label = sym(r.feature, fallback=r.symbol).replace('$', '$')
    print(f'{label} & {r.mean_abs_shap:.4f} & {r.share_pct:.2f}\\% & '
          f'{r.cumulative_pct:.2f}\\% \\\\')
print(f'% top three = {rank.share_pct.head(3).sum():.2f} %,  '
      f'top twelve cumulative = {rank.cumulative_pct.iloc[NTOP-1]:.2f} %')

---
### Output

Everything is written to `figures/redrawn/`:

| File | Replaces |
|---|---|
| `figure2.pdf` / `.png` | Fig. 2 (whole figure) |
| `figure3abc.pdf` / `.png` | the **a–c** column of Fig. 3 — compose with the unchanged **d–e** |
| `figure4a.pdf` / `.png` | panel **a** of Fig. 4 — compose with the unchanged **b–d** |
| `SI_3.pdf` / `.png` | SI Fig. S3, now with the decile-bias panel |

Nothing under `plotdata/`, `archive/` or the manuscript directory is modified.
